# Cas12a manuscript / sgRNA Modeler: create and export fixed split

Place this notebook in the root of the cloned `cas12a_manuscript` repository.

The manuscript repository delegates machine-learning work to the separate
`sgrna_modeler` package. This notebook auto-locates the packaged Kim 2018
AsCas12a training dataset:

`sgrna_modeler/sgrna_modeler/data/datasets/Kim_2018_Train.csv.zip`

The split follows your standard protocol:

- NumPy seed = 42
- exactly 12,000 samples selected as the seen pool
- 80% training and 20% validation within the seen pool
- all remaining samples used as unseen

The exact split is saved and then loaded by the companion 100-trial notebook.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42
SELECTED_SEEN_SIZE = 12000
VALIDATION_FRACTION = 0.20

# Expected layout:
# cas12a_manuscript/
# ├── Cas12a_create_and_export_fixed_split.ipynb
# └── sgrna_modeler/
#     └── sgrna_modeler/data/datasets/Kim_2018_Train.csv.zip

DATA_CANDIDATES = [
    Path("sgrna_modeler/sgrna_modeler/data/datasets/Kim_2018_Train.csv.zip"),
    Path("../sgrna_modeler/sgrna_modeler/data/datasets/Kim_2018_Train.csv.zip"),
]

DATA_FILE = next(
    (path for path in DATA_CANDIDATES if path.exists()),
    None,
)

if DATA_FILE is None:
    checked = "\n".join(str(path.resolve()) for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "Could not find Kim_2018_Train.csv.zip. Checked:\n"
        f"{checked}\n"
        "Clone sgrna_modeler inside or beside cas12a_manuscript."
    )

BASE_OUTPUT_DIR = Path("results/cas12a_kim2018_fixed_split")
SPLIT_DIR = BASE_OUTPUT_DIR / "saved_splits"
TXT_DIR = BASE_OUTPUT_DIR / "split_sequences"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TXT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_FILE.resolve())
print("Output:", BASE_OUTPUT_DIR.resolve())


In [ ]:
raw_data = pd.read_csv(DATA_FILE)

print("Raw shape:", raw_data.shape)
print("Columns:")
for column in raw_data.columns:
    print(" ", repr(column))

SEQUENCE_COLUMN = "Context Sequence"
ACTIVITY_COLUMN = "Indel frequency"

for required_column in [SEQUENCE_COLUMN, ACTIVITY_COLUMN]:
    if required_column not in raw_data.columns:
        raise ValueError(
            f"Required column {required_column!r} was not found."
        )

data = raw_data[
    [SEQUENCE_COLUMN, ACTIVITY_COLUMN]
].copy()

data = data.rename(
    columns={
        SEQUENCE_COLUMN: "target_context_sequence",
        ACTIVITY_COLUMN: "activity",
    }
)

data["target_context_sequence"] = (
    data["target_context_sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

data["activity"] = pd.to_numeric(
    data["activity"],
    errors="coerce",
)

# Convert activity from 0–100 to 0–1
data["activity"] = data["activity"] / 100.0

valid_sequence = data[
    "target_context_sequence"
].str.fullmatch("[ACGT]+")

valid_activity = np.isfinite(
    data["activity"].to_numpy(dtype=float)
)

data = (
    data.loc[valid_sequence & valid_activity]
    .drop_duplicates()
    .reset_index(drop=True)
)

sequence_lengths = (
    data["target_context_sequence"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("Cleaned rows:", len(data))
print("Sequence lengths:")
print(sequence_lengths)

# Keep only the dominant context length if multiple lengths exist.
dominant_length = int(sequence_lengths.idxmax())
data = data.loc[
    data["target_context_sequence"].str.len() == dominant_length
].reset_index(drop=True)

data["sample_id"] = np.arange(len(data), dtype=int)
data["context_length"] = dominant_length

if len(data) <= SELECTED_SEEN_SIZE:
    raise ValueError(
        f"Need more than {SELECTED_SEEN_SIZE} valid samples; "
        f"found {len(data)}."
    )

print("Dominant context length:", dominant_length)
print("Final samples:", len(data))
display(data.head())


In [ ]:
np.random.seed(SPLIT_SEED)

full_indices = np.arange(len(data))

selected_indices = np.random.choice(
    len(full_indices),
    size=SELECTED_SEEN_SIZE,
    replace=False,
)

unseen_indices = np.setdiff1d(
    full_indices,
    selected_indices,
)

train_indices, validation_indices = train_test_split(
    selected_indices,
    test_size=VALIDATION_FRACTION,
    random_state=SPLIT_SEED,
)

train_data = data.iloc[train_indices].reset_index(drop=True)
validation_data = data.iloc[validation_indices].reset_index(drop=True)
unseen_data = data.iloc[unseen_indices].reset_index(drop=True)

assert set(train_data["sample_id"]).isdisjoint(
    validation_data["sample_id"]
)
assert set(train_data["sample_id"]).isdisjoint(
    unseen_data["sample_id"]
)
assert set(validation_data["sample_id"]).isdisjoint(
    unseen_data["sample_id"]
)

display(pd.DataFrame({
    "subset": ["train", "validation", "unseen"],
    "n_rows": [
        len(train_data),
        len(validation_data),
        len(unseen_data),
    ],
    "fraction": [
        len(train_data) / len(data),
        len(validation_data) / len(data),
        len(unseen_data) / len(data),
    ],
}))


In [ ]:
train_file = SPLIT_DIR / "train_split.csv"
validation_file = SPLIT_DIR / "validation_split.csv"
unseen_file = SPLIT_DIR / "unseen_split.csv"

train_data.to_csv(train_file, index=False)
validation_data.to_csv(validation_file, index=False)
unseen_data.to_csv(unseen_file, index=False)

np.savetxt(
    BASE_OUTPUT_DIR / "train_indices.txt",
    train_indices,
    fmt="%d",
)
np.savetxt(
    BASE_OUTPUT_DIR / "validation_indices.txt",
    validation_indices,
    fmt="%d",
)
np.savetxt(
    BASE_OUTPUT_DIR / "unseen_indices.txt",
    unseen_indices,
    fmt="%d",
)

def export_subset(frame, subset_name):
    subset_dir = TXT_DIR / subset_name
    subset_dir.mkdir(parents=True, exist_ok=True)

    frame["sample_id"].to_csv(
        subset_dir / f"{subset_name}_sample_id.txt",
        index=False,
        header=False,
    )

    frame["target_context_sequence"].to_csv(
        subset_dir / f"{subset_name}_target_context.txt",
        index=False,
        header=False,
    )

    frame["activity"].to_csv(
        subset_dir / f"{subset_name}_activity.txt",
        index=False,
        header=False,
    )

    frame.to_csv(
        subset_dir / f"{subset_name}_complete.tsv",
        sep="\t",
        index=False,
    )

export_subset(train_data, "train")
export_subset(validation_data, "validation")
export_subset(unseen_data, "unseen")

print("Saved split tables and TXT exports.")


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "source_dataset": "Kim_2018_Train",
    "source_file": str(DATA_FILE),
    "source_sha256": sha256(DATA_FILE),
    "sequence_column": SEQUENCE_COLUMN,
    "activity_column": ACTIVITY_COLUMN,
    "context_length": int(dominant_length),
    "split_seed": SPLIT_SEED,
    "selected_seen_size": SELECTED_SEEN_SIZE,
    "validation_fraction_within_seen": VALIDATION_FRACTION,
    "n_total": int(len(data)),
    "n_train": int(len(train_data)),
    "n_validation": int(len(validation_data)),
    "n_unseen": int(len(unseen_data)),
    "train_sha256": sha256(train_file),
    "validation_sha256": sha256(validation_file),
    "unseen_sha256": sha256(unseen_file),
}

with open(BASE_OUTPUT_DIR / "split_manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2)

print("Manifest saved.")
